# 05 - Gold Aggregation
Business-level KPIs computed from Silver, written as Gold Delta tables.

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "sentinel_catalog.sentinel_schema.silver_transactions"
GOLD_KPI_TABLE = "sentinel_catalog.sentinel_schema.gold_kpi_summary"
GOLD_BANK_TABLE = "sentinel_catalog.sentinel_schema.gold_bank_metrics"

silver_df = spark.table(SILVER_TABLE).select(
    "transaction_id", "event_timestamp", "bank_name", "amount", "status"
)  # column pruning: only what KPIs need

### Overall KPI summary

In [0]:
kpi_df = silver_df.agg(
    F.count("*").alias("total_transactions"),
    F.sum("amount").alias("total_volume"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.sum(F.when(F.col("status") == "SUCCESS", 1).otherwise(0)).alias("successful_transactions"),
    F.sum(F.when(F.col("status") == "FAILED", 1).otherwise(0)).alias("failed_transactions"),
    F.sum(F.when(F.col("status") == "TIMEOUT", 1).otherwise(0)).alias("timeout_transactions"),
    F.sum(F.when(F.col("status") == "PENDING", 1).otherwise(0)).alias("pending_transactions"),
).withColumn(
    "failure_rate",
    F.round((F.col("failed_transactions") + F.col("timeout_transactions")) / F.col("total_transactions"), 4)
).withColumn("generated_at", F.current_timestamp())

kpi_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_KPI_TABLE)
display(kpi_df)

total_transactions,total_volume,avg_amount,successful_transactions,failed_transactions,timeout_transactions,pending_transactions,failure_rate,generated_at
14580,5.0252573794999963E8,34466.79,3646,3537,3965,3432,0.5145,2026-08-08T17:46:55.811Z


### Bank-level and time-based metrics

In [0]:
bank_df = (
    silver_df.groupBy("bank_name")
    .agg(
        F.count("*").alias("txn_count"),
        F.sum("amount").alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.sum(F.when(F.col("status") == "SUCCESS", 1).otherwise(0)).alias("success_count"),
        F.sum(F.when(F.col("status") == "FAILED", 1).otherwise(0)).alias("failed_count"),
        F.sum(F.when(F.col("status") == "TIMEOUT", 1).otherwise(0)).alias("timeout_count"),
    )
    .withColumn("failure_rate", F.round((F.col("failed_count") + F.col("timeout_count")) / F.col("txn_count"), 4))
)

hourly_df = (
    silver_df.withColumn("txn_hour", F.date_trunc("hour", "event_timestamp"))
    .groupBy("txn_hour")
    .agg(F.count("*").alias("txn_count"), F.sum("amount").alias("volume"))
    .orderBy("txn_hour")
)

bank_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_BANK_TABLE)

display(bank_df.orderBy(F.desc("total_amount")))
display(hourly_df)

bank_name,txn_count,total_amount,avg_amount,success_count,failed_count,timeout_count,failure_rate
YES_BANK,2115,7.639727210999992E7,36121.64,533,521,561,0.5116
AXIS,2098,7.301830340000013E7,34803.77,545,471,562,0.4924
ICICI,2096,7.133863923999995E7,34035.61,527,504,564,0.5095
KOTAK,2027,7.12945197900001E7,35172.43,514,479,568,0.5165
PNB,2099,7.076659722E7,33714.43,478,554,589,0.5445
SBI,2065,7.03471444599999E7,34066.41,519,489,571,0.5133
HDFC,2080,6.936326173000012E7,33347.72,530,519,550,0.5139


txn_hour,txn_count,volume
2026-08-01T00:00:00.000Z,488,1.733275056E7
2026-08-01T01:00:00.000Z,485,1.6336296199999996E7
2026-08-01T02:00:00.000Z,484,1.5462324079999987E7
2026-08-01T03:00:00.000Z,485,1.6523302490000008E7
2026-08-01T04:00:00.000Z,485,1.7753893080000002E7
2026-08-01T05:00:00.000Z,486,1.6012915960000003E7
2026-08-01T06:00:00.000Z,489,1.7136708659999993E7
2026-08-01T07:00:00.000Z,480,1.7626954709999986E7
2026-08-01T08:00:00.000Z,486,1.474134169E7
2026-08-01T09:00:00.000Z,487,1.665361101E7
